# Chuẩn bị dữ liệu thực nghiệm

Notebook này chuẩn bị hai nguồn dữ liệu cho các thí nghiệm Machine Learning và Rule-based Scoring.

Nguyên tắc thực hiện:

- Hai nguồn được xử lý và chia độc lập, không ghép theo từng sinh viên.
- Nguồn 1 chỉ giữ `Dropout` và `Graduate`; tạm loại `Enrolled`.
- Nguồn 2 giữ nguyên biến mục tiêu nhị phân `Dropout`.
- Mỗi nguồn được chia theo tỷ lệ 70% huấn luyện, 15% xác thực và 15% kiểm thử.
- Dùng `random_state = 42` và chia phân tầng theo biến mục tiêu để có thể tái lập kết quả.
- Notebook chưa điền giá trị thiếu, mã hóa hay chuẩn hóa. Các thao tác đó phải được học từ tập huấn luyện ở bước xây dựng mô hình.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# Hiển thị đủ các cột và hạn chế việc bảng kết quả bị xuống dòng khó đọc.
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

# Hỗ trợ chạy notebook từ thư mục notebooks hoặc từ thư mục gốc của kho mã.
data_candidates = [Path("../data"), Path("data")]
DATA_DIR = next((path for path in data_candidates if path.exists()), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục data. Hãy chạy notebook từ thư mục notebooks "
        "hoặc thư mục gốc của kho mã."
    )

# Tạo thư mục lưu file chia dữ liệu nếu thư mục này chưa tồn tại.
SPLIT_DIR = DATA_DIR.parent / "outputs" / "splits"
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_1_PATH = DATA_DIR / "student_dropout_and_success.csv"
SOURCE_2_PATH = DATA_DIR / "student_dropout.csv"

# Các hằng số này phải giống nhau trong mọi lần chạy thí nghiệm.
RANDOM_STATE = 42
TEST_RATIO = 0.15
VALIDATION_RATIO = 0.15

print("Thư mục dữ liệu:", DATA_DIR.resolve())
print("Thư mục lưu các tập đã chia:", SPLIT_DIR.resolve())

## 1. Đọc và kiểm tra hai nguồn dữ liệu

In [ ]:
# Đọc riêng từng file để bảo đảm hai nguồn không bị ghép nhầm với nhau.
source_1_raw = pd.read_csv(SOURCE_1_PATH)
source_2_raw = pd.read_csv(SOURCE_2_PATH)

# Dừng sớm nếu file đầu vào bị thay đổi cấu trúc so với kết quả EDA.
assert source_1_raw.shape == (4424, 35), (
    f"Kích thước Nguồn 1 không đúng: {source_1_raw.shape}"
)
assert source_2_raw.shape == (10000, 19), (
    f"Kích thước Nguồn 2 không đúng: {source_2_raw.shape}"
)
assert set(source_1_raw["Target"].dropna().unique()) == {
    "Dropout", "Graduate", "Enrolled"
}, "Nguồn 1 xuất hiện nhãn mục tiêu ngoài dự kiến."
assert set(source_2_raw["Dropout"].dropna().unique()) == {0, 1}, (
    "Nguồn 2 xuất hiện nhãn mục tiêu ngoài 0 và 1."
)

print("Nguồn 1 — student_dropout_and_success.csv:", source_1_raw.shape)
print("Nguồn 2 — student_dropout.csv:", source_2_raw.shape)

## 2. Chuẩn hóa biến mục tiêu

`row_id` được tạo từ số thứ tự dòng ban đầu. Cột này giúp tất cả giải pháp đọc đúng cùng một tập dữ liệu và đối chiếu kết quả dự đoán về sau.

In [ ]:
# Lưu số thứ tự dòng gốc trước khi lọc để có thể truy vết về file ban đầu.
source_1_with_id = source_1_raw.reset_index(names="row_id")
source_2 = source_2_raw.reset_index(names="row_id").copy()

# Nguồn 1: chỉ giữ các sinh viên đã có kết quả cuối cùng.
source_1 = source_1_with_id[
    source_1_with_id["Target"].isin(["Dropout", "Graduate"])
].copy()

# Quy ước chung: bỏ học = 1, không bỏ học/tốt nghiệp = 0.
source_1["Dropout"] = (source_1["Target"] == "Dropout").astype(int)
source_2["Dropout"] = source_2["Dropout"].astype(int)

# Đếm các dòng Enrolled bị loại để nhóm có thể trình bày minh bạch trong báo cáo.
excluded_enrolled = int((source_1_with_id["Target"] == "Enrolled").sum())

assert len(source_1) == 3630
assert excluded_enrolled == 794
assert source_1["row_id"].is_unique and source_2["row_id"].is_unique

print("Nguồn 1 còn lại sau khi loại Enrolled:", len(source_1))
print("Số dòng Enrolled tạm loại:", excluded_enrolled)
print("Nguồn 2 được giữ nguyên:", len(source_2))

## 3. Hàm chia dữ liệu 70% / 15% / 15%

Việc chia được thực hiện hai lần:

1. Tách 15% dữ liệu thành tập kiểm thử.
2. Từ 85% còn lại, tách một phần tương ứng 15% toàn bộ dữ liệu thành tập xác thực.
3. Phần còn lại trở thành tập huấn luyện, tương đương khoảng 70% toàn bộ dữ liệu.

In [ ]:
def create_stratified_split(frame, target_column):
    """Chia dữ liệu phân tầng và trả về bảng ánh xạ row_id sang từng tập."""

    # Lần chia thứ nhất: giữ riêng 15% làm tập kiểm thử cuối cùng.
    train_validation, test = train_test_split(
        frame,
        test_size=TEST_RATIO,
        random_state=RANDOM_STATE,
        stratify=frame[target_column],
    )

    # Quy đổi 15% toàn bộ dữ liệu thành tỷ lệ tương ứng trong 85% còn lại.
    validation_ratio_in_remainder = VALIDATION_RATIO / (1 - TEST_RATIO)

    # Lần chia thứ hai: tách tập xác thực khỏi phần huấn luyện + xác thực.
    train, validation = train_test_split(
        train_validation,
        test_size=validation_ratio_in_remainder,
        random_state=RANDOM_STATE,
        stratify=train_validation[target_column],
    )

    # Tạo một bảng duy nhất để các giải pháp có thể dùng chung cách chia.
    split_map = pd.concat(
        [
            train[["row_id", target_column]].assign(split="train"),
            validation[["row_id", target_column]].assign(split="validation"),
            test[["row_id", target_column]].assign(split="test"),
        ],
        ignore_index=True,
    )

    # Đổi tên nhãn thành target để hai file ánh xạ có cùng cấu trúc.
    split_map = split_map.rename(columns={target_column: "target"})
    return split_map[["row_id", "split", "target"]].sort_values("row_id")


## 4. Chia độc lập từng nguồn và kiểm tra tính hợp lệ

In [ ]:
# Hai lời gọi riêng biệt tạo ra hai phép chia độc lập cho hai nguồn dữ liệu.
source_1_split = create_stratified_split(source_1, "Dropout")
source_2_split = create_stratified_split(source_2, "Dropout")

def validate_split(frame, split_map, source_name):
    """Kiểm tra mỗi dòng chỉ thuộc một tập và không bị thất thoát khi chia."""

    # Mỗi row_id phải xuất hiện đúng một lần trong bảng ánh xạ.
    assert split_map["row_id"].is_unique, f"{source_name}: row_id bị trùng."
    assert set(split_map["row_id"]) == set(frame["row_id"]), (
        f"{source_name}: có dòng bị thiếu hoặc thừa sau khi chia."
    )
    assert set(split_map["split"]) == {"train", "validation", "test"}

    # Ba tập không được chứa cùng một row_id.
    row_sets = {
        split_name: set(split_map.loc[split_map["split"] == split_name, "row_id"])
        for split_name in ["train", "validation", "test"]
    }
    assert row_sets["train"].isdisjoint(row_sets["validation"])
    assert row_sets["train"].isdisjoint(row_sets["test"])
    assert row_sets["validation"].isdisjoint(row_sets["test"])

validate_split(source_1, source_1_split, "Nguồn 1")
validate_split(source_2, source_2_split, "Nguồn 2")

print("Đã kiểm tra: không có dòng bị thiếu, bị trùng hoặc xuất hiện ở nhiều tập.")

## 5. Kiểm tra số lượng và tỷ lệ nhãn

Bảng dưới đây giúp xác nhận tỷ lệ bỏ học được duy trì gần giống nhau trong ba tập nhờ phương pháp chia phân tầng.

In [ ]:
def summarize_split(split_map, source_name):
    """Tóm tắt kích thước và tỷ lệ lớp bỏ học trong từng tập."""

    # Tính số dòng, số ca bỏ học và tỷ lệ bỏ học của từng tập.
    summary = (
        split_map.groupby("split", as_index=False)
        .agg(so_dong=("row_id", "size"), so_ca_bo_hoc=("target", "sum"))
    )
    summary["ty_le_du_lieu_phan_tram"] = (
        summary["so_dong"] / len(split_map) * 100
    ).round(2)
    summary["ty_le_bo_hoc_phan_tram"] = (
        summary["so_ca_bo_hoc"] / summary["so_dong"] * 100
    ).round(2)
    summary.insert(0, "nguon_du_lieu", source_name)
    return summary

split_summary = pd.concat(
    [
        summarize_split(source_1_split, "student_dropout_and_success.csv"),
        summarize_split(source_2_split, "student_dropout.csv"),
    ],
    ignore_index=True,
)

print(split_summary.to_string(index=False))

## 6. Xuất hai file ánh xạ bắt buộc và sáu tập dữ liệu

Hai file `*_split.csv` là nguồn quy ước chính. Cả Machine Learning và Rule-based Scoring phải đọc các file này thay vì tự chia lại dữ liệu. Sáu file `*_train.csv`, `*_validation.csv`, `*_test.csv` được xuất thêm để nhóm có thể sử dụng trực tiếp.

In [ ]:
# Xuất bảng ánh xạ chính thức để mọi giải pháp dùng cùng row_id và cùng tập kiểm thử.
source_1_split.to_csv(
    SPLIT_DIR / "student_dropout_and_success_split.csv",
    index=False,
    encoding="utf-8-sig",
)
source_2_split.to_csv(
    SPLIT_DIR / "student_dropout_split.csv",
    index=False,
    encoding="utf-8-sig",
)

def export_materialized_splits(frame, split_map, file_prefix):
    """Ghép theo row_id để xuất dữ liệu đầy đủ của từng tập đã chia."""

    # Ghép ở đây chỉ là ghép bảng ánh xạ với chính nguồn của nó, không phải ghép hai nguồn.
    frame_with_split = frame.merge(
        split_map[["row_id", "split"]],
        on="row_id",
        how="inner",
        validate="one_to_one",
    )

    # Xuất riêng ba tập và giữ row_id để có thể đối chiếu dự đoán sau này.
    for split_name in ["train", "validation", "test"]:
        split_frame = (
            frame_with_split.loc[frame_with_split["split"] == split_name]
            .drop(columns="split")
            .sort_values("row_id")
        )
        split_frame.to_csv(
            SPLIT_DIR / f"{file_prefix}_{split_name}.csv",
            index=False,
            encoding="utf-8-sig",
        )

export_materialized_splits(
    source_1, source_1_split, "student_dropout_and_success"
)
export_materialized_splits(source_2, source_2_split, "student_dropout")

# Lưu bảng tóm tắt để nhóm dùng khi viết báo cáo và kiểm tra nhanh.
split_summary.to_csv(
    SPLIT_DIR / "split_summary.csv", index=False, encoding="utf-8-sig"
)

print("Đã xuất các file sau:")
for output_path in sorted(SPLIT_DIR.glob("*.csv")):
    print("-", output_path)

## 7. Kết luận

Sau bước này, hai nguồn đã có cách chia dữ liệu cố định và có thể tái lập. Machine Learning và Rule-based Scoring phải dùng đúng `row_id` trong hai file ánh xạ. Tập kiểm thử chỉ dùng một lần để báo cáo kết quả cuối cùng; không dùng tập này để chọn đặc trưng, xây luật, điều chỉnh trọng số, chọn ngưỡng hoặc chọn siêu tham số.